# MonIA Kaggle — ONE CLICK
Run All. This notebook generates two candidate-only clips for surprise gameplay. GitHub Actions can inject a gameplay job into the worker bundle; otherwise the test job is used. Shot 2 starts from the actual last frame of shot 1 to preserve continuity. Nothing is approved or published into live manifests automatically.

In [ ]:
%pip -q install -U diffusers transformers accelerate safetensors imageio[ffmpeg] huggingface_hub requests ftfy "pillow==11.3.0"
print('✅ Dependencies ready')

In [ ]:
import os, json, requests, torch, PIL, imageio.v3 as iio
from pathlib import Path
from diffusers.utils import export_to_video, load_image
print('Pillow:', PIL.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU first')
REPO='vartcom38-collab/marion-lucas-game'
RAW=f'https://raw.githubusercontent.com/{REPO}/main'
WORK=Path('/kaggle/working/monia-studio'); WORK.mkdir(parents=True, exist_ok=True)
print('✅ MonIA one-click ready')

In [ ]:
def get_json(url):
    r=requests.get(url, timeout=30); r.raise_for_status(); return r.json()
def download(url, target):
    r=requests.get(url, timeout=120); r.raise_for_status(); Path(target).write_bytes(r.content); return str(target)
bundle_job=Path('job.json')
if bundle_job.exists():
    job=json.loads(bundle_job.read_text(encoding='utf-8'))
    print('✅ Dynamic gameplay job loaded from worker bundle')
else:
    job=get_json(RAW + '/studio/queue/test-kaggle-marion-001.json')
    print('ℹ️ No dynamic job bundled; using repository test job')
assert job.get('candidateOnly') is True and job.get('narrativeAuthority') is False
job_id=job['id']; job_dir=WORK/job_id; job_dir.mkdir(parents=True, exist_ok=True)
refs={}
for ch in job.get('characters', []):
    refs[ch['id']]=download(ch['canonRef'], job_dir/f"canon-{ch['id']}.jpg")
print('✅ Job loaded:', job_id)

In [ ]:
from diffusers import LTXImageToVideoPipeline
g=job.get('generation') or {}
print('⏳ Loading LTX model...')
pipe=LTXImageToVideoPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
primary=job.get('primaryCharacter') or ('lucas' if 'lucas' in refs else 'marion' if 'marion' in refs else next(iter(refs.keys())))
image=load_image(refs[primary])
base_seed=int(g.get('seed',240907))
clips=[]
continuity='generated-image'
for index,seed in enumerate((base_seed,base_seed+1),start=1):
    name=f'shot-{index:02d}.mp4'
    print(f'🎬 Generating surprise candidate {index}/2 from {continuity}...')
    generator=torch.Generator(device='cpu').manual_seed(seed)
    frames=pipe(image=image,prompt=job['prompt'],negative_prompt=job.get('negativePrompt'),width=int(g.get('width',320)),height=int(g.get('height',512)),num_frames=int(g.get('frames',17)),num_inference_steps=int(g.get('steps',8)),generator=generator).frames[0]
    out=str(job_dir/name)
    export_to_video(frames,out,fps=int(g.get('fps',12)))
    clips.append(name)
    print('✅ Candidate generated:',out)
    if index==1:
        last=iio.imread(out,index=-1,plugin='pyav')
        frame_path=job_dir/'continuity-shot-01.png'
        iio.imwrite(frame_path,last)
        image=load_image(str(frame_path))
        continuity='previous-video-frame'

In [ ]:
result={'jobId':job_id,'state':'candidate','candidateOnly':True,'narrativeAuthority':False,'router':'ltx','selectionMode':'surprise-auto','continuity':['generated-image','previous-video-frame'],'clips':clips}
(job_dir/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print('✅ Two-candidate continuity package ready for GitHub Actions pickup')
print('✅ DONE — surprise candidates only, never auto-approved into manifests')